# 03 — Notification Policy Simulation
Sweeps decision threshold 0.1 → 0.9 and analyses the operational trade-off:
- alerts/week vs missed failures
- prevented failures vs false service calls
- mean lead-time CDF at default threshold

**Expected property:** higher threshold → fewer alerts → more missed failures (monotone trade-off).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from evaluation.splits import track_a_split, track_b_split
from models.baseline.logistic import LogisticBaseline
from models.classical.rf_xgb import train_xgboost
from policy.notification import sweep_threshold, lead_time_cdf

print('Imports OK')

## 1. Train a quick model (no HPO) for simulation

In [ ]:
ta = track_a_split('FD001')

# Fast XGBoost without HPO for policy simulation
xgb = train_xgboost(ta.train, ta.feature_cols, run_hpo=False)
X_test = ta.test[ta.feature_cols].fillna(0).values
y_score = xgb.predict_proba(X_test)[:, 1]
y_true  = ta.test['label_within_x'].values
rul     = ta.test['rul'].values

print(f'Test samples: {len(y_true)}')
print(f'Positive rate: {y_true.mean():.1%}')

## 2. Threshold sweep

In [ ]:
sweep = sweep_threshold(y_true, y_score, rul)
sweep

## 3. Trade-off plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Alerts vs missed failures ---
ax = axes[0]
ax2 = ax.twinx()
ax.plot(sweep['threshold'], sweep['alerts'], 'o-', color='steelblue', label='Alerts')
ax2.plot(sweep['threshold'], sweep['missed_failures'], 's--', color='firebrick', label='Missed failures')
ax.set_xlabel('Threshold')
ax.set_ylabel('Alerts', color='steelblue')
ax2.set_ylabel('Missed failures', color='firebrick')
ax.set_title('Alerts vs Missed Failures')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2)

# --- Prevented vs false calls ---
ax = axes[1]
ax.plot(sweep['threshold'], sweep['prevented_failures'], 'o-', color='green', label='Prevented')
ax.plot(sweep['threshold'], sweep['false_service_calls'], 's--', color='orange', label='False calls')
ax.set_xlabel('Threshold')
ax.set_ylabel('Count')
ax.set_title('Prevented Failures vs False Service Calls')
ax.legend()

# --- Mean lead time ---
ax = axes[2]
ax.plot(sweep['threshold'], sweep['mean_lead_time_cycles'], 'o-', color='purple')
ax.set_xlabel('Threshold')
ax.set_ylabel('Mean lead time (cycles)')
ax.set_title('Mean Lead Time of Prevented Failures')

plt.suptitle('Notification Policy Threshold Sweep — FD001', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Lead-time CDF at default threshold (0.5)

In [ ]:
cdf_df = lead_time_cdf(y_true, y_score, rul, threshold=0.5)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cdf_df['lead_time_cycles'], cdf_df['cdf'], color='steelblue', lw=2)
ax.axvline(30, color='firebrick', linestyle='--', label='30-cycle horizon')
ax.set_xlabel('Lead time (cycles before failure)')
ax.set_ylabel('CDF (fraction of prevented failures)')
ax.set_title('Lead Time CDF — threshold=0.5, FD001')
ax.legend()
plt.tight_layout()
plt.show()

p50_lead = cdf_df.loc[cdf_df['cdf'] >= 0.50, 'lead_time_cycles'].iloc[0] if len(cdf_df) else 0
print(f'Median lead time (p50): {p50_lead:.0f} cycles')

## 5. Business metric table — Track A and Track B at threshold=0.5

In [ ]:
from evaluation.metrics import business_metrics

tb = track_b_split()
X_tb = tb.test[[c for c in ta.feature_cols if c in tb.test.columns]].fillna(0).values
y_score_tb = xgb.predict_proba(X_tb)[:, 1]

ta_biz = business_metrics(y_true, y_score, threshold=0.5)
tb_biz = business_metrics(tb.test['label_within_x'].values, y_score_tb, threshold=0.5)

biz_df = pd.DataFrame([ta_biz, tb_biz], index=['Track A (FD001 holdout)', 'Track B (FD002+FD004)'])
print('\nBusiness Metrics at threshold=0.5:')
biz_df

## 6. Monotone trade-off verification

In [ ]:
# Verify: higher threshold → fewer alerts (non-increasing)
alerts = sweep['alerts'].values
missed = sweep['missed_failures'].values
assert all(alerts[i] >= alerts[i+1] for i in range(len(alerts)-1)), 'Alerts not monotone!'
assert all(missed[i] <= missed[i+1] for i in range(len(missed)-1)), 'Missed not monotone!'
print('✓ Monotone trade-off verified: higher threshold → fewer alerts, more missed failures')